In [86]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,SystemMessage
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator
import os
import asyncio
from tavily import AsyncTavilyClient

In [87]:
load_dotenv()

True

In [88]:
model = ChatOpenAI(model='gpt-4o-mini')


In [89]:
class topic_analyser_schema(BaseModel):
    topic_summary : str = Field(description='Summary of the topic')
    web_research_questions :list[str] = Field(description='Web search question')
    statistics_questions :list[str] = Field(description='statistics questions')
    expert_opinion_questions :list[str] = Field(description='expert opinion questions')
    report_sections :list[str] = Field(description='report sections')

In [90]:
structure_topic_analyser = model.with_structured_output(topic_analyser_schema)

In [ ]:
class AgentState(TypedDict):
    topic: str
    topic_summary :str
    web_research_questions :list[str]
    statistics_questions :list[str]
    expert_opinion_questions :list[str]
    report_sections :list[str]
    web_research: list[object]
    statistics: str
    web_findings: list[dict]
    stats_findings: list[dict]
    expert_findings: list[dict]
    expert_opinions: str
    merged_research: str
    report: str
    quality_score: int
    iteration_count: int

In [92]:
def topic_analyser(state: AgentState):

    messages = [
        SystemMessage(
            content="""
You are an expert Research Planning Agent.

Your responsibility is to analyze a topic and create a structured research plan.

Do NOT perform research.

Generate:
- concise topic summary
- web research questions
- statistics research questions
- expert opinion questions
- report sections

Ensure all questions are:
- specific
- non-overlapping
- researchable
- relevant
"""
        ),
        HumanMessage(
            content=f"Topic: {state['topic']}"
        )
    ]

    response = structure_topic_analyser.invoke(messages)

    return response.model_dump()

In [93]:
client = AsyncTavilyClient(os.getenv('TAVILY_API_KEY'))

In [94]:
async def search_all(state:AgentState):
    questions = state['web_research_questions']
    results = await asyncio.gather(*[client.search(q) for q in questions])
    return results


In [ ]:
async def web_search(state:AgentState):
    results = await search_all(state)
    # print(results)
    
    return {'web_research':results}
# await web_search(AgentState)
    

In [ ]:
Research Synthesis Node

In [96]:
graph = StateGraph(AgentState)
graph.add_node('topic_analyser',topic_analyser)
graph.add_node('web_search',web_search)
graph.add_edge(START,'topic_analyser')
graph.add_edge('topic_analyser','web_search')
graph.add_edge('web_search',END)
workflow = graph.compile()

In [97]:
initialState = {
    'topic' : "Impact of Ai on software industry"
}
final_state =await workflow.ainvoke(initialState)
print (final_state)

{'topic': 'Impact of Ai on software industry', 'topic_summary': 'The impact of AI on the software industry refers to how advancements in artificial intelligence technologies are transforming software development processes, enhancing automation, improving user experiences, and influencing business models. AI is reshaping coding practices, testing, project management, and product delivery, creating opportunities for increased efficiency and innovation, while also raising concerns around job displacement and ethical considerations.', 'web_research_questions': ['What are the key trends in AI integration within the software industry?', 'How has the adoption of AI tools changed the software development lifecycle?', 'What are the leading AI technologies currently being used in software development?', 'What challenges do software companies face when implementing AI solutions?', 'How do companies measure the ROI of AI in their software processes?'], 'statistics_questions': ['What percentage of 